# Part 1 : La Régression Linéaire

**Durée estimée : 2h30**

## Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** l'intuition derrière la régression linéaire (tracer la "meilleure ligne")
2. **Interpréter** les coefficients d'un modèle de régression
3. **Maîtriser** les hyperparamètres de LinearRegression
4. **Appliquer** la régression linéaire avec scikit-learn sur des données réelles
5. **Évaluer** la performance avec les métriques de régression (Ch2 Part3)
6. **Identifier** quand la régression linéaire est appropriée (et quand elle ne l'est pas)

---

## Problème Réel : Comment Zillow estime-t-il le prix de votre maison ?

Imaginez : vous voulez vendre votre appartement. Avant même de contacter une agence, vous ouvrez **Zillow** (ou SeLoger en France) et en quelques secondes, le site vous donne une estimation : **245 000 €**.

**Question :** Comment le site peut-il connaître la valeur de VOTRE appartement, qu'il n'a jamais visité ?

*(Prenez 30 secondes pour réfléchir avant de lire la suite...)*

<details>
<summary>Votre intuition ?</summary>

### Réponse

Zillow utilise les **ventes passées** de propriétés similaires pour prédire le prix de la vôtre. Si des appartements de 50m² dans votre quartier se sont vendus autour de 5 000 €/m², alors votre appartement de 50m² vaut probablement environ 250 000 €.

C'est exactement ce que fait la **régression linéaire** : trouver une relation mathématique entre des caractéristiques (surface, nombre de pièces, quartier) et un résultat (prix).

</details>

### Le Zestimate de Zillow : un cas d'étude réel

Selon [Zillow Research](https://www.zillow.com/research/zestimate-forecast-methodology/), leur algorithme **Zestimate** analyse **7,5 millions de modèles statistiques** pour estimer la valeur de plus de 110 millions de propriétés aux États-Unis.

Le cœur de leur approche ? La **régression** : prédire un nombre (le prix) à partir de caractéristiques connues.

| Caractéristique | Impact sur le prix |
|-----------------|-------------------|
| Surface (m²) | +5 000 €/m² |
| Nombre de chambres | +15 000 €/chambre |
| Distance métro | -2 000 €/km |
| Année de construction | Variable |

**Question :** Si chaque caractéristique a un "impact" sur le prix, comment les combiner pour obtenir une estimation finale ?

*(Réponse attendue : On additionne les impacts de chaque caractéristique)*

---

## 1.1 Intuition : Tracer la "Meilleure Ligne"

Commençons par le cas le plus simple : **une seule caractéristique**.

Imaginons que vous avez les données de 5 appartements vendus récemment :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Données de 5 appartements
surface = np.array([30, 45, 60, 75, 90])  # en m²
prix = np.array([150, 210, 280, 350, 430])  # en milliers d'euros

# Visualisation
plt.figure(figsize=(10, 6))
plt.scatter(surface, prix, s=100, c='blue', label='Appartements vendus')
plt.xlabel('Surface (m²)', fontsize=12)
plt.ylabel('Prix (k€)', fontsize=12)
plt.title('Prix des appartements selon leur surface', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

**Question Socratique :** En regardant ce graphique, voyez-vous une tendance ? Si un ami vous demandait d'estimer le prix d'un appartement de 50 m², comment feriez-vous ?

*(Réponse attendue : Les points semblent alignés. Je tracerais mentalement une ligne et je regarderais où 50 m² tombe sur cette ligne — probablement autour de 240-250 k€)*

### La ligne qui "passe au mieux" par les points

Votre intuition est exactement ce que fait la régression linéaire : **trouver la ligne qui passe au mieux par tous les points**.

Mais comment définir "au mieux" ?

In [ ]:
# Trois lignes candidates
plt.figure(figsize=(12, 5))

# Données
plt.scatter(surface, prix, s=100, c='blue', zorder=5)

# Ligne 1 : trop basse
x_line = np.linspace(20, 100, 100)
plt.plot(x_line, 3*x_line + 80, 'r--', label='Ligne 1 : trop basse', alpha=0.7)

# Ligne 2 : trop haute  
plt.plot(x_line, 5*x_line + 50, 'g--', label='Ligne 2 : trop haute', alpha=0.7)

# Ligne 3 : juste bien (approximation)
plt.plot(x_line, 4.7*x_line + 12, 'orange', linewidth=2, label='Ligne 3 : optimale ?')

plt.xlabel('Surface (m²)', fontsize=12)
plt.ylabel('Prix (k€)', fontsize=12)
plt.title('Quelle ligne "passe au mieux" par les points ?', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>Question Socratique : Comment mesurer si une ligne est "bonne" ?</summary>

### Réponse

On mesure la **distance entre chaque point et la ligne**. Une bonne ligne minimise ces distances.

Plus précisément, on mesure la **distance verticale** (l'erreur de prédiction) : pour chaque appartement, on compare le prix **réel** au prix **prédit** par la ligne.

```
Erreur = Prix réel - Prix prédit
```

La meilleure ligne est celle qui minimise la somme de toutes ces erreurs (au carré, pour pénaliser les grosses erreurs).

</details>

---

## 1.2 Construction : Les Moindres Carrés et l'API sklearn

### L'équation de la droite

Toute droite peut s'écrire sous la forme :

```
y = w × x + b
```

Où :
- **y** (La Cible) = ce qu'on prédit (le prix)
- **x** (La Feature) = la donnée d'entrée (la surface)
- **w** (Le Coefficient / Poids) = la **pente**, combien y augmente quand x augmente de 1
- **b** (L'Ordonnée à l'origine / Biais) = valeur de y quand x = 0

```
┌─────────────────────────────────────────────────────────────┐
│     Prix (y)                                                │
│       ▲                              ╱                      │
│       │                           ╱                         │
│       │                        ╱   ← pente (w) = combien    │
│       │                     ╱        le prix augmente       │
│       │                  ╱           par m² supplémentaire  │
│       │               ╱                                     │
│     b ├──────╱   ← ordonnée à l'origine (prix de base)      │
│       │    ╱                                                │
│       └────────────────────────────▶ Surface (x)            │
└─────────────────────────────────────────────────────────────┘
```

### Pourquoi "moindres carrés" ?

**Question :** Si on additionne simplement les erreurs, quel problème aurait-on ?

*(Réponse : Les erreurs positives et négatives s'annuleraient ! +10 et -10 = 0, alors que le modèle est faux)*

**Solution :** Mettre les erreurs **au carré** avant de les additionner → Les **moindres carrés**.

In [ ]:
# Transformation obligatoire pour Scikit-Learn : 2D
X = surface.reshape(-1, 1)  # (5, 1) au lieu de (5,)
y = prix

# Entraînement (sur toutes les données pour la démo)
model = LinearRegression()
model.fit(X, y)

# ⚠️ ATTENTION : En vrai projet, JAMAIS fit() sur toutes les données !
# On fait ici pour comprendre comment la ligne se trace.

print(f"Formule trouvée : Prix = {model.coef_[0]:.2f} × Surface + {model.intercept_:.2f}")

In [ ]:
# Visualiser les erreurs (résidus)
prix_predit = model.predict(X)

plt.figure(figsize=(12, 6))
plt.scatter(surface, prix, s=100, c='blue', zorder=5, label='Prix réel')
plt.plot(surface, prix_predit, 'orange', linewidth=2, label='Ligne de régression')

# Tracer les erreurs (résidus)
for i in range(len(surface)):
    plt.plot([surface[i], surface[i]], [prix[i], prix_predit[i]], 
             'r-', linewidth=2, alpha=0.7)

plt.xlabel('Surface (m²)', fontsize=12)
plt.ylabel('Prix (k€)', fontsize=12)
plt.title('Les erreurs (résidus) : distance entre réalité et prédiction', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Somme des erreurs au carré : {np.sum((prix - prix_predit)**2):.2f}")

### Interprétation des coefficients

<details>
<summary>Question Socratique : Que signifie concrètement une pente de 4.67 ?</summary>

### Réponse

Une pente de **4.67 k€/m²** signifie que :

> Pour chaque mètre carré supplémentaire, le prix augmente d'environ **4 670 €**.

C'est l'**interprétation du coefficient** — une des forces majeures de la régression linéaire : les coefficients ont un **sens business direct**.

</details>

In [ ]:
# Afficher les paramètres du modèle
print("═" * 50)
print("PARAMÈTRES DU MODÈLE DE RÉGRESSION")
print("═" * 50)
print(f"\n  Pente (w)            : {model.coef_[0]:.2f} k€/m²")
print(f"  Ordonnée à l'origine : {model.intercept_:.2f} k€")
print(f"\n  Équation : Prix = {model.coef_[0]:.2f} × Surface + {model.intercept_:.2f}")
print("\n" + "═" * 50)

In [ ]:
# Prédire un nouveau prix
nouvelle_surface = 55
prix_estime = model.predict([[nouvelle_surface]])[0]

print(f"\nEstimation pour un appartement de {nouvelle_surface} m² :")
print(f"   Prix = {model.coef_[0]:.2f} × {nouvelle_surface} + {model.intercept_:.2f}")
print(f"   Prix = {prix_estime:.1f} k€ (soit environ {prix_estime * 1000:.0f} €)")

### La Régression Multiple : plusieurs caractéristiques

Dans la réalité, le prix dépend de **plusieurs** caractéristiques :

```
Prix = w₁ × Surface + w₂ × Chambres + w₃ × Distance_métro + ... + b
```

Chaque caractéristique a son propre **coefficient** → C'est **l'interprétabilité**.

In [ ]:
# Créer un dataset plus réaliste
np.random.seed(42)
n = 100

data = pd.DataFrame({
    'surface': np.random.uniform(25, 120, n),
    'chambres': np.random.randint(1, 5, n),
    'etage': np.random.randint(0, 10, n),
    'distance_metro': np.random.uniform(0.1, 3, n),
    'annee_construction': np.random.randint(1960, 2023, n)
})

# Prix selon formule "vraie" + bruit
data['prix'] = (
    4.5 * data['surface'] +
    25 * data['chambres'] +
    3 * data['etage'] +
    -15 * data['distance_metro'] +
    0.5 * (data['annee_construction'] - 1960) +
    np.random.normal(0, 20, n)
)

print("Aperçu des données :")
data.head()

In [ ]:
# Séparer features et target + Split train/test
X = data.drop('prix', axis=1)
y = data['prix']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entraîner le modèle
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

In [ ]:
# Afficher les coefficients
print("\n" + "═" * 60)
print("COEFFICIENTS DU MODÈLE DE RÉGRESSION MULTIPLE")
print("═" * 60)

coef_df = pd.DataFrame({
    'Caractéristique': X.columns,
    'Coefficient': model_multi.coef_,
    'Interprétation': [
        f"+{model_multi.coef_[0]:.1f} k€ par m² supplémentaire",
        f"+{model_multi.coef_[1]:.1f} k€ par chambre supplémentaire",
        f"+{model_multi.coef_[2]:.1f} k€ par étage supplémentaire",
        f"{model_multi.coef_[3]:.1f} k€ par km du métro (négatif !)",
        f"+{model_multi.coef_[4]:.2f} k€ par année de construction"
    ]
})

print(coef_df.to_string(index=False))
print(f"\nOrdonnée à l'origine (b) : {model_multi.intercept_:.2f} k€")
print("═" * 60)

In [ ]:
# Visualiser l'importance des coefficients
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in model_multi.coef_]
plt.barh(X.columns, model_multi.coef_, color=colors)
plt.xlabel('Coefficient (impact sur le prix en k€)', fontsize=12)
plt.title('Impact de chaque caractéristique sur le prix', fontsize=14)
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---

## 1.3 Hyperparamètres : Contrôler LinearRegression

**Rappel (Ch2 Part3)** : Les hyperparamètres sont des choix qu'on fait AVANT l'entraînement. Ils ne sont PAS appris par le modèle.

La régression linéaire classique a **peu d'hyperparamètres** (contrairement à d'autres algorithmes), mais il est important de les connaître.

```
┌─────────────────────────────────────────────────────────────────────┐
│           HYPERPARAMÈTRES DE LinearRegression                      │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   PARAM         │ DEFAULT  │ DESCRIPTION                           │
│   ─────────────────────────────────────────────────────────────────│
│   fit_intercept │ True     │ Calcule l'ordonnée à l'origine (b)    │
│   copy_X        │ True     │ Copie X avant fit (sécurité)          │
│   n_jobs        │ None     │ Parallélisation (-1 = tous CPU)       │
│   positive      │ False    │ Force les coefficients ≥ 0            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### fit_intercept : Inclure ou non l'ordonnée à l'origine

**fit_intercept=True** (défaut) : La droite peut croiser l'axe Y n'importe où.

**fit_intercept=False** : La droite DOIT passer par l'origine (0, 0).

**Quand utiliser False ?** Rarement ! Seulement si vous savez que votre relation passe par (0, 0). Ex: "0 heure travaillée = 0 € de salaire".

In [ ]:
# Comparaison : avec vs sans intercept
X_demo = surface.reshape(-1, 1)
y_demo = prix

# Avec intercept (défaut)
model_avec = LinearRegression(fit_intercept=True)
model_avec.fit(X_demo, y_demo)

# Sans intercept (forcé à passer par 0)
model_sans = LinearRegression(fit_intercept=False)
model_sans.fit(X_demo, y_demo)

# Visualisation
plt.figure(figsize=(12, 5))
x_line = np.linspace(0, 100, 100).reshape(-1, 1)

plt.scatter(surface, prix, s=100, c='blue', zorder=5, label='Données')
plt.plot(x_line, model_avec.predict(x_line), 'orange', linewidth=2, 
         label=f'fit_intercept=True (b={model_avec.intercept_:.1f})')
plt.plot(x_line, model_sans.predict(x_line), 'green', linewidth=2, linestyle='--',
         label=f'fit_intercept=False (b=0, forcé)')
plt.scatter([0], [0], s=100, c='red', marker='x', zorder=10, label='Origine (0,0)')

plt.xlabel('Surface (m²)')
plt.ylabel('Prix (k€)')
plt.title('Impact de fit_intercept')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Avec intercept : Score R² = {model_avec.score(X_demo, y_demo):.4f}")
print(f"Sans intercept : Score R² = {model_sans.score(X_demo, y_demo):.4f}")

### positive : Forcer des coefficients positifs

**positive=True** : Tous les coefficients doivent être ≥ 0.

**Quand l'utiliser ?** Quand vous savez que la relation ne peut être que positive. Ex: Plus de sucre → Plus de calories (jamais moins).

In [ ]:
# Exemple avec positive=True
# Rappel : distance_metro a un coefficient négatif naturellement

model_libre = LinearRegression()
model_libre.fit(X_train, y_train)

model_positif = LinearRegression(positive=True)
model_positif.fit(X_train, y_train)

print("═" * 60)
print("COMPARAISON : COEFFICIENTS LIBRES vs FORCÉS POSITIFS")
print("═" * 60)

comparaison = pd.DataFrame({
    'Feature': X.columns,
    'Coef Libre': model_libre.coef_.round(2),
    'Coef Positif': model_positif.coef_.round(2)
})
print(comparaison.to_string(index=False))
print("\n⚠️ Attention : positive=True force distance_metro à 0 !")
print("   Cela perd de l'information. À utiliser avec précaution.")
print("═" * 60)

### n_jobs : Parallélisation

Pour les très gros datasets, **n_jobs=-1** utilise tous vos CPU pour accélérer le calcul.

```python
# Pour les gros datasets
model = LinearRegression(n_jobs=-1)
```

### Résumé des hyperparamètres

| Hyperparamètre | Défaut | Quand changer ? |
|----------------|--------|------------------|
| fit_intercept | True | Rarement (si passage par 0 garanti) |
| positive | False | Si tous les effets doivent être positifs |
| n_jobs | None | -1 pour gros datasets |

---

## 1.4 Évaluation : Appliquer les métriques de régression

**Rappel (Ch2 Part3)** : Vous avez appris les métriques de régression :
- **MAE** : Erreur moyenne absolue (interprétable)
- **RMSE** : Pénalise les grosses erreurs
- **R²** : % de variance expliquée

Appliquons-les maintenant à notre modèle immobilier.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fonction utilitaire de Ch2 Part3
def evaluer_regression(y_vrai, y_pred, nom_modele="Modèle"):
    """Affiche toutes les métriques de régression."""
    mae = mean_absolute_error(y_vrai, y_pred)
    rmse = np.sqrt(mean_squared_error(y_vrai, y_pred))
    r2 = r2_score(y_vrai, y_pred)
    
    print(f"\nÉvaluation : {nom_modele}")
    print("=" * 50)
    print(f"MAE  : {mae:,.2f}")
    print(f"RMSE : {rmse:,.2f}")
    print(f"R²   : {r2:.4f} ({r2*100:.1f}% de variance expliquée)")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

In [ ]:
# Évaluation sur TRAIN et TEST
y_pred_train = model_multi.predict(X_train)
y_pred_test = model_multi.predict(X_test)

print("═" * 55)
print("ÉVALUATION DU MODÈLE IMMOBILIER")
print("═" * 55)

metrics_train = evaluer_regression(y_train, y_pred_train, "Train set")
metrics_test = evaluer_regression(y_test, y_pred_test, "Test set")

### Diagnostic : Train vs Test

**Rappel (Ch2 Part3)** : Comparer les performances train/test révèle :
- **Train >> Test** : Overfitting (le modèle mémorise)
- **Train ≈ Test** : Bonne généralisation

In [ ]:
# Diagnostic visuel
gap_r2 = metrics_train['R2'] - metrics_test['R2']

print("\n" + "═" * 55)
print("DIAGNOSTIC TRAIN vs TEST")
print("═" * 55)
print(f"R² Train : {metrics_train['R2']:.4f}")
print(f"R² Test  : {metrics_test['R2']:.4f}")
print(f"Gap      : {gap_r2:.4f}")

if gap_r2 < 0.05:
    print("\n✅ Bonne généralisation ! Le gap est faible.")
elif gap_r2 < 0.15:
    print("\n⚠️ Gap modéré. Surveillez l'overfitting.")
else:
    print("\n❌ Overfitting probable ! Le modèle mémorise.")

In [ ]:
# Visualisation des résidus (diagnostic qualité)
residus = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Résidus vs Prédictions
axes[0].scatter(y_pred_test, residus, alpha=0.6)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Prix prédit (k€)')
axes[0].set_ylabel('Résidu (erreur)')
axes[0].set_title('Résidus vs Prédictions\n(Idéal : nuage aléatoire autour de 0)')

# Distribution des résidus
axes[1].hist(residus, bins=15, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Résidu')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des résidus\n(Idéal : cloche centrée sur 0)')

plt.tight_layout()
plt.show()

---

## 1.5 Limites : Quand NE PAS utiliser la régression linéaire

La régression linéaire est puissante, mais elle repose sur des **hypothèses**.

### Hypothèse 1 : La relation doit être LINÉAIRE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cas 1 : Relation linéaire
x1 = np.linspace(0, 10, 50)
y1 = 2*x1 + 3 + np.random.normal(0, 1.5, 50)
axes[0].scatter(x1, y1, alpha=0.7)
axes[0].plot(x1, 2*x1 + 3, 'r-', linewidth=2, label='Régression linéaire')
axes[0].set_title('✅ Cas A : Relation linéaire', fontsize=12)
axes[0].legend()

# Cas 2 : Relation non-linéaire
x2 = np.linspace(0, 10, 50)
y2 = 0.5*x2**2 + np.random.normal(0, 2, 50)
axes[1].scatter(x2, y2, alpha=0.7)
coef = np.polyfit(x2, y2, 1)
axes[1].plot(x2, coef[0]*x2 + coef[1], 'r-', linewidth=2, label='Régression linéaire')
axes[1].plot(x2, 0.5*x2**2, 'g--', linewidth=2, label='Vraie relation (courbe)')
axes[1].set_title('❌ Cas B : Relation NON-linéaire', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

### Hypothèse 2 : Pas de multicollinéarité

**Multicollinéarité** = deux caractéristiques très corrélées entre elles.

**Exemple problématique :** Surface en m² ET Surface en pieds carrés (juste une conversion !).

→ Les coefficients deviennent instables et difficiles à interpréter.

### Hypothèse 3 : Homoscédasticité

Les erreurs doivent avoir une **variance constante** (pas de "trompette").

- **Bon scénario** : Que l'appartement fasse 20 m² ou 200 m², le modèle se trompe de ~5 000 €
- **Mauvais scénario** : Précis sur les studios (erreur 1 000 €), imprécis sur les villas (erreur 200 000 €)

### Résumé : Quand utiliser (ou éviter) la régression linéaire

| Situation | Régression linéaire ? |
|-----------|----------------------|
| Prédire un prix, un coût, une température | ✅ Oui |
| Relation clairement linéaire | ✅ Oui |
| Besoin d'interpréter les coefficients | ✅ Oui |
| Relation en forme de U ou de courbe | ❌ Non |
| Variables très corrélées entre elles | ⚠️ Attention |
| Prédire une catégorie (oui/non, spam/pas spam) | ❌ Non → Régression logistique (Part 2) |

---

## Exercice Pratique : California Housing

Appliquez tout ce que vous avez appris !

In [ ]:
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing()
X_cal = pd.DataFrame(california.data, columns=california.feature_names)
y_cal = california.target * 100  # Prix en milliers de $

print("Dataset California Housing")
print(f"Nombre d'observations : {len(X_cal)}")
print(f"\nCaractéristiques : {list(X_cal.columns)}")
X_cal.head()

### Votre mission :

1. Split train/test (80/20)
2. Entraîner LinearRegression
3. Afficher les coefficients (quel facteur impacte le plus ?)
4. Évaluer avec evaluer_regression() sur train et test
5. Y a-t-il de l'overfitting ?

In [ ]:
# À VOUS DE JOUER !

# Étape 1 : Split
# X_train, X_test, y_train, y_test = ...

# Étape 2 : Entraîner
# model = ...

# Étape 3 : Coefficients
# ...

# Étape 4 : Évaluer
# ...

# Étape 5 : Diagnostic

In [ ]:
# SOLUTION

# Étape 1 : Split
X_train_cal, X_test_cal, y_train_cal, y_test_cal = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)

# Étape 2 : Entraîner
model_cal = LinearRegression()
model_cal.fit(X_train_cal, y_train_cal)

# Étape 3 : Coefficients
coefs = pd.DataFrame({
    'Feature': X_cal.columns,
    'Coefficient': model_cal.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("Coefficients triés par importance :")
print(coefs.to_string(index=False))

# Étape 4 : Évaluer
y_pred_train_cal = model_cal.predict(X_train_cal)
y_pred_test_cal = model_cal.predict(X_test_cal)

m_train = evaluer_regression(y_train_cal, y_pred_train_cal, "California - Train")
m_test = evaluer_regression(y_test_cal, y_pred_test_cal, "California - Test")

# Étape 5 : Diagnostic
gap = m_train['R2'] - m_test['R2']
print(f"\nGap R² (train - test) : {gap:.4f}")
print("✅ Pas d'overfitting majeur" if gap < 0.05 else "⚠️ Léger overfitting")

---

## Récapitulatif

### Structure de cette partie :

| Section | Contenu |
|---------|--------|
| **Hook** | Zillow Zestimate - estimation immobilière |
| **1.1 Intuition** | La "meilleure ligne" qui passe par les points |
| **1.2 Construction** | Moindres carrés, équation y = wx + b, coefficients |
| **1.3 Hyperparamètres** | fit_intercept, positive, n_jobs |
| **1.4 Évaluation** | MAE, RMSE, R² (appliqués via evaluer_regression) |
| **1.5 Limites** | Linéarité, multicollinéarité, homoscédasticité |

### Code essentiel :

```python
from sklearn.linear_model import LinearRegression

# Entraîner
model = LinearRegression()
model.fit(X_train, y_train)

# Paramètres appris
print(model.coef_)        # Coefficients (pentes)
print(model.intercept_)   # Ordonnée à l'origine

# Prédire et évaluer
predictions = model.predict(X_test)
evaluer_regression(y_test, predictions)
```

### Prochaine partie : Régression Logistique

Et si on voulait prédire non pas un **nombre** mais une **catégorie** (spam/pas spam) ? C'est le sujet de la Part 2 !

---

## Réflexion Métacognitive

Avant de passer à la suite :

1. Pouvez-vous expliquer ce que signifie un coefficient de régression à quelqu'un ?
2. Quel hyperparamètre changeriez-vous si vos coefficients devaient tous être positifs ?
3. Comment diagnostiquez-vous l'overfitting en régression ?
4. Dans votre domaine, voyez-vous une application de la régression linéaire ?

---

**Sources :**
- [Zillow Zestimate Forecast Methodology](https://www.zillow.com/research/zestimate-forecast-methodology/)
- [scikit-learn LinearRegression Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [Emerald 2024 - Housing Price Prediction Study](https://www.emerald.com/insight/content/doi/10.1108/ijhma-09-2023-0120/full/html)